In [2]:
# ==========================================================
#      Modelos Predictivos de PM2.5 con Validación Cruzada
# ==========================================================
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error
import joblib
import os


In [3]:
# ==========================================================
# 1. Función para evaluar el modelo
# ==========================================================
def evaluar_modelo(modelo, X_test, y_test):
    predicciones = modelo.predict(X_test)
    predicciones = np.where(predicciones > 0, predicciones, np.nan)  # descartar negativos

    # eliminar NaN si los hay
    mask = ~np.isnan(predicciones)
    predicciones = predicciones[mask]
    y_test = y_test[mask]

    r2 = r2_score(y_test, predicciones)
    pearson = np.corrcoef(y_test, predicciones)[0, 1]
    rmse = np.sqrt(mean_squared_error(y_test, predicciones))
    bias = np.mean(predicciones - y_test)

    resultados = pd.DataFrame({
        'R2': [round(r2, 5)],
        'Pearson': [round(pearson, 3)],
        'RMSE': [round(rmse, 3)],
        'Bias': [round(bias, 3)],
        'Min_Pred': [round(np.min(predicciones), 3)],
        'Max_Pred': [round(np.max(predicciones), 3)]
    })
    return resultados


In [3]:
# ----------------------------------------------------------
# 1. Cargar datos
# ----------------------------------------------------------
estacion = "MX"
modelo = "1"
base_dir = f"D:/Josefina/Proyectos/ProyectoChile/{estacion}/modelos/ParticionDataSet/"
train_data = pd.read_csv(f"{base_dir}/Modelo_{modelo}/M{modelo}_train_{estacion}.csv")
test_data  = pd.read_csv(f"{base_dir}/Modelo_{modelo}/M{modelo}_test_{estacion}.csv")

X_train = train_data[["AOD_055",'ndvi', 'BCSMASS_dia', 'DUSMASS_dia', 'SO2SMASS_dia', 'SO4SMASS_dia',
                      'SSSMASS_dia', 'blh_mean', 'd2m_mean', 't2m_mean',
                      'v10_mean', 'u10_mean', 'tp_mean', 'DEM', 'dayWeek']]#'sp_mean', 
y_train = train_data['PM25']

X_test = test_data[X_train.columns]
y_test = test_data['PM25']

# ----------------------------------------------------------
# 2. Escalado de datos
# ----------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ----------------------------------------------------------
# 3. Definir búsqueda limitada de hiperparámetros. Revisar esto!!
# ----------------------------------------------------------
# param_grid = {
#     'C': [0.1, 1, 10, 100],
#     'gamma': ['scale', 0.1, 0.01, 0.001],
#     'epsilon': [0.01, 0.1, 0.2]
# }

# grid_search = GridSearchCV(
#     SVR(),
#     param_grid=param_grid,
#     cv=10,
#     scoring='r2',
#     n_jobs=-1
# )



param_dist = {
    'C': [0.1, 1, 10],          # 3 valores
    'gamma': ['scale', 0.1, 0.01],  # 3 valores
    'epsilon': [0.01, 0.1]      # 2 valores
}


cv = KFold(n_splits=10, shuffle=True, random_state=123)
svr = SVR(kernel='rbf')

random_search = RandomizedSearchCV(
    estimator=svr,
    param_distributions=param_dist,
    n_iter=5,        # probar 5 combinaciones aleatorias (equivale a tuneLength=5)
    cv=cv,
    scoring='r2',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# # ----------------------------------------------------------
# # 4. Entrenamiento del modelo
# # ----------------------------------------------------------
random_search.fit(X_train_scaled, y_train)
best_model = random_search.best_estimator_

# # ----------------------------------------------------------
# # 5. Evaluación del modelo
# # ----------------------------------------------------------
resultados_SVR_cv = evaluar_modelo(best_model, X_test_scaled, y_test)
print(resultados_SVR_cv)

# # ----------------------------------------------------------
# # 6. Guardar modelo entrenado
# # ----------------------------------------------------------
# output_dir = f"D:/Josefina/Proyectos/Tesis/{estacion}/modelos/"
# os.makedirs(output_dir, exist_ok=True)
# joblib.dump(best_model, f"{output_dir}/01-SVR-CV-M{modelo}-AOD-{estacion}.pkl")

# # ----------------------------------------------------------
# # 7. Mejor combinación de hiperparámetros
# # ----------------------------------------------------------
# print("Hiperparámetros óptimos:")
# print(random_search.best_params_)


# SVR      R2  Pearson   RMSE   Bias  Min_Pred  Max_Pred (py)       R2    RMSE   Bias (R)
# SP: 0.56941    0.773  7.096 -0.939     2.146    44.239     /// 0.69, 6.06, -0.68,
# "ST": 0.64816    0.822  9.767 -1.529     3.502    61.251  /// 0.82, 7.12, -0.53,
# BA  0.40788    0.695  8.173 -1.729     5.158    36.071 //// 0.50, 7.52, -0.37
# MD 0.48163     0.71  6.272 -0.917     4.924    43.433  //// 0.78, 4.17, -0.44 ///// 
#MX 0.50816    0.726  6.977 -0.916     1.993    48.591 //// 0.67, 5.76, -0.58,

Fitting 10 folds for each of 5 candidates, totalling 50 fits
        R2  Pearson   RMSE   Bias  Min_Pred  Max_Pred
0  0.50816    0.726  6.977 -0.916     1.993    48.591


#####      Extra Trees Regressor con CV ##### 

In [3]:
# ==========================================================
#      Extra Trees Regressor con CV
# ==========================================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, cross_val_predict


# ----------------------------------------------------------
# 1. Cargar datos
# ----------------------------------------------------------
estacion = "CH"
modelo = "1"
base_dir = f"D:/Josefina/Proyectos/ProyectoChile/{estacion}/modelos/ParticionDataSet/"
train_data = pd.read_csv(f"{base_dir}/Modelo_{modelo}/M{modelo}_train_{estacion}.csv")
test_data  = pd.read_csv(f"{base_dir}/Modelo_{modelo}/M{modelo}_test_{estacion}.csv")

X_train = train_data[['AOD_055', 'ndvi', 'BCSMASS_dia', 'DUSMASS_dia', 'SO2SMASS_dia',
                      'SO4SMASS_dia','SSSMASS_dia', 'blh_mean',  'd2m_mean', #'sp_mean',
                      't2m_mean','v10_mean', 'u10_mean', 'tp_mean', 'DEM', 'dayWeek']]
y_train = train_data['PM25']

X_test = test_data[X_train.columns]
y_test = test_data['PM25']

# ----------------------------------------------------------
# 2. Escalado opcional (ET no requiere pero si quieres estandarizar)
# ----------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ----------------------------------------------------------
# 3. Definir modelo Extra Trees con hiperparámetros fijos
# ----------------------------------------------------------
modelo_ET = ExtraTreesRegressor(
    n_estimators=500,         # número de árboles, se puede cambiar
    max_features=5,           # equivalente a mtry
    min_samples_leaf=5,       # equivalente a min.node.size
    bootstrap=False,          # Extra Trees usa muestra completa
    criterion='mse',          # equivalente a 'impurity'
    random_state=123,
    n_jobs=-1
)

# ----------------------------------------------------------
# 4. Validación cruzada 10-fold (opcional)
# ----------------------------------------------------------
cv = KFold(n_splits=10, shuffle=True, random_state=123)
pred_cv = cross_val_predict(modelo_ET, X_train_scaled, y_train, cv=cv, n_jobs=-1)

# Entrenamiento final sobre todo el set de entrenamiento
modelo_ET.fit(X_train_scaled, y_train)

# ----------------------------------------------------------
# 5. Evaluación en test
# ----------------------------------------------------------
resultados_ET_cv = evaluar_modelo(modelo_ET, X_test_scaled, y_test)
print(resultados_ET_cv)

# ----------------------------------------------------------
# 6. Guardar modelo
# ----------------------------------------------------------
# output_dir = f"D:/Josefina/Proyectos/Tesis/{estacion}/modelos/"
# os.makedirs(output_dir, exist_ok=True)
# joblib.dump(modelo_ET, f"{output_dir}/01-ET-CV-M{modelo}-{estacion}.pkl")

# ----------------------------------------------------------
# 7. Resumen de hiperparámetros
# ----------------------------------------------------------
print("Número de árboles:", modelo_ET.n_estimators)
print("Mtry / max_features:", modelo_ET.max_features)
print("Min node size / min_samples_leaf:", modelo_ET.min_samples_leaf)
print("Criterio:", modelo_ET.criterion)

# ET     R2  Pearson   RMSE   Bias  Min_Pred  Max_Pred (py)       R2    RMSE   Bias (R)
# "SP",  0.67046    0.839  6.207 -0.048     7.784    50.288 /// 0.72, 5.92, 0.10,
#"ST", 0.78622    0.893  7.613 -0.001     7.516    80.876 /////0.83, 6.91, 0.16,
# "BA", 0.54475    0.761  7.166  0.094     6.154    47.839 //// 0.59, 7.02, 0.30,
#"MD",  0.65863     0.83  5.09 -0.014     6.018    53.515 ////  0.87, 3.40, 0.03,
#"MX", 0.64499    0.823  5.928  0.057      7.23    74.287 ////  0.72, 5.52, 0.17

        R2  Pearson   RMSE   Bias  Min_Pred  Max_Pred
0  0.64499    0.823  5.928  0.057      7.23    74.287
Número de árboles: 500
Mtry / max_features: 5
Min node size / min_samples_leaf: 5
Criterio: mse


In [3]:
# ----------------------------------------
# Random Forest con CV=10 en Python
# ----------------------------------------
# import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import r2_score, mean_squared_error
# import numpy as np
import time

# Función para calcular métricas como tu evaluar_modelo
def evaluar_modelo(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_pred > 0  # filtramos predicciones negativas si hace falta
    y_true, y_pred = y_true[mask], y_pred[mask]
    
    r2 = r2_score(y_true, y_pred)
    pearson = np.corrcoef(y_true, y_pred)[0,1]
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)
    
    resultados = {
        'R2': round(r2, 5),
        'Pearson': round(pearson, 3),
        'RMSE': round(rmse, 3),
        'Bias': round(bias, 3),
        'Min_Pred': round(np.min(y_pred), 3),
        'Max_Pred': round(np.max(y_pred), 3)
    }
    return resultados

# -------------------------------
# Cargar datos
# -------------------------------
estacion = "MX"
modelo = "1"
dir_base = f"D:/Josefina/Proyectos/ProyectoChile/{estacion}/modelos/ParticionDataSet/"

train_data = pd.read_csv(f"{dir_base}Modelo_{modelo}/M{modelo}_train_{estacion}.csv")
test_data = pd.read_csv(f"{dir_base}Modelo_{modelo}/M{modelo}_test_{estacion}.csv")

X_train = train_data[['AOD_055','ndvi','BCSMASS_dia','DUSMASS_dia','SO4SMASS_dia',
                      'v10_mean','SSSMASS_dia','blh_mean','sp_mean','SO2SMASS_dia',
                      'd2m_mean','tp_mean','DEM','t2m_mean']]

y_train = train_data['PM25']

X_test = test_data[['AOD_055','ndvi','BCSMASS_dia','DUSMASS_dia','SO4SMASS_dia',
                    'v10_mean','SSSMASS_dia','blh_mean','sp_mean','SO2SMASS_dia',
                    'd2m_mean','tp_mean','DEM','t2m_mean']]

y_test = test_data['PM25']

# -------------------------------
# Definir Random Forest y Grid de hiperparámetros
# -------------------------------
rf = RandomForestRegressor(random_state=123)

param_grid = {
    'n_estimators': [100],      # Número de árboles
    'max_features': [5],        # Equivalente a mtry
    'min_samples_leaf': [5],    # Tamaño mínimo de nodo terminal
    'max_depth': [None]         # opcional: equivalente a maxnodes
}

# -------------------------------
# Entrenamiento con CV=10
# -------------------------------
cv = 10
start_time = time.time()

grid_rf = GridSearchCV(estimator=rf, param_grid=param_grid,
                       cv=cv, n_jobs=-1, verbose=2, scoring='r2')

grid_rf.fit(X_train, y_train)

end_time = time.time()
print(f"Tiempo de entrenamiento: {end_time - start_time:.2f} s")

# -------------------------------
# Resultados
# -------------------------------
print("Mejores hiperparámetros:", grid_rf.best_params_)

# Predicciones sobre el set de testeo
y_pred = grid_rf.predict(X_test)
resultados_RF_cv = evaluar_modelo(y_test, y_pred)
print(resultados_RF_cv)

# Parámetros del modelo final
final_rf = grid_rf.best_estimator_
print("Número de árboles:", final_rf.n_estimators)
print("Max features (mtry):", final_rf.max_features)
print("Min samples leaf (nodesize):", final_rf.min_samples_leaf)
print("Max depth:", final_rf.max_depth)



#"SP", R2': 0.62719, 'Pearson': 0.802, 'RMSE': 6.602, 'Bias': 0.039, 'Min_Pred': 7.045,  56.467 /// 0.71, 5.94, 0.11,
# "ST",  0.76218,  0.874,  8.03,  0.078,  8.096,  88.143/// 0.84, 6.66, 0.12,
# "BA"  0.47482,  0.704,  7.697,  0.057,  5.372, 55.252/// "RF", 0.57, 7.04, 0.28,
# "MD", 0.5362,  0.74,  5.933,  0.08, 5.454, 51.598 /// 0.73, 4.57, 0.09,
# "MX",  0.60636, 0.784, 6.242, 0.036, 6.302, 76.28 /// 0.52, 5.42, 0.15


Fitting 10 folds for each of 1 candidates, totalling 10 fits
Tiempo de entrenamiento: 26.56 s
Mejores hiperparámetros: {'max_depth': None, 'max_features': 5, 'min_samples_leaf': 5, 'n_estimators': 100}
{'R2': 0.60636, 'Pearson': 0.784, 'RMSE': 6.242, 'Bias': 0.036, 'Min_Pred': 6.302, 'Max_Pred': 76.28}
Número de árboles: 100
Max features (mtry): 5
Min samples leaf (nodesize): 5
Max depth: None


In [ ]:
###2da version RF

In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1️⃣ FUNCIÓN PARA EVALUAR EL MODELO
# ------------------------------------------------------------
def evaluar_modelo(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_pred > 0  # filtramos predicciones negativas si hace falta
    y_true, y_pred = y_true[mask], y_pred[mask]
    
    r2 = r2_score(y_true, y_pred)
    pearson = np.corrcoef(y_true, y_pred)[0,1]
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)
    
    resultados = {
        'R2': round(r2, 5),
        'Pearson': round(pearson, 3),
        'RMSE': round(rmse, 3),
        'Bias': round(bias, 3),
        'Min_Pred': round(np.min(y_pred), 3),
        'Max_Pred': round(np.max(y_pred), 3)
    }
    return resultados

# ------------------------------------------------------------
# 2️⃣ CARGA DE DATOS
# ------------------------------------------------------------
estacion = "MD"
modelo = "1"
dir_base = f"D:/Josefina/Proyectos/ProyectoChile/{estacion}/modelos/ParticionDataSet/"

train_data = pd.read_csv(f"{dir_base}Modelo_{modelo}/M{modelo}_train_{estacion}.csv")
test_data = pd.read_csv(f"{dir_base}Modelo_{modelo}/M{modelo}_test_{estacion}.csv")

# Variables predictoras y objetivo
vars_predictoras = [
    'AOD_055','ndvi','BCSMASS_dia','DUSMASS_dia','SO4SMASS_dia',
    'v10_mean','SSSMASS_dia','blh_mean','sp_mean','SO2SMASS_dia',
    'd2m_mean','tp_mean','DEM','t2m_mean'
]

X_train = train_data[vars_predictoras]
y_train = train_data['PM25']

X_test = test_data[vars_predictoras]
y_test = test_data['PM25']

# ------------------------------------------------------------
# 3️⃣ CONFIGURACIÓN DE RANDOM FOREST + GRID SEARCH
# ------------------------------------------------------------
rf = RandomForestRegressor(
    n_estimators=500,
    bootstrap=True,
    random_state=123,
    n_jobs=-1
)

# ParamGrid equivalente a tuneGrid de caret
param_grid = {
    'max_features': [3, 5, 7],       # mtry en R
    'min_samples_leaf': [1, 3, 5],   # nodesize aproximado
}

# Validación cruzada de 10 pliegues (como trainControl)
cv = KFold(n_splits=10, shuffle=True, random_state=123)

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

# ------------------------------------------------------------
# 4️⃣ ENTRENAMIENTO
# ------------------------------------------------------------
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_

print("\n📊 Mejor combinación de hiperparámetros:")
print(grid_rf.best_params_)

# ------------------------------------------------------------
# 5️⃣ PREDICCIÓN Y EVALUACIÓN
# ------------------------------------------------------------
# Evaluación en train
y_train_pred = best_rf.predict(X_train)
train_metrics = evaluar_modelo(y_train, y_train_pred)

# Evaluación en test
y_test_pred = best_rf.predict(X_test)
test_metrics = evaluar_modelo(y_test, y_test_pred)

print("\n✅ Métricas en TRAIN:")
for k, v in train_metrics.items():
    print(f"{k}: {v}")

print("\n✅ Métricas en TEST:")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

# # ------------------------------------------------------------
# # 6️⃣ IMPORTANCIA DE VARIABLES
# # ------------------------------------------------------------
# importancia = pd.DataFrame({
#     'Variable': X_train.columns,
#     'Importancia': best_rf.feature_importances_
# }).sort_values(by='Importancia', ascending=False)

# print("\n🌲 Importancia de variables:")
# print(importancia)

# # (opcional) guardar resultados
# importancia.to_csv(f"{dir_base}importancia_variables_M{modelo}_{estacion}.csv", index=False)


# R2, RMSE, Bias
# 1ERA PRUEBA ---- R ----2DA PRUEBA
#"SP": 0.62719, 6.602, 0.039 -------- 0.71, 5.94, 0.11 -------- 0.66392,  6.269,0.168,
# "ST",  0.76218,   8.03,  0.078 -------- 0.84, 6.66, 0.12 -------- 0.78665,  7.605, 0.233
# "BA"  0.47482,    7.697,  0.057 --------  0.57, 7.04, 0.28 -------- 0.48642, 7.611, 0.088
# "MD", 0.5362,   5.933,  0.08 -------- 0.73, 4.57, 0.09 -------- 0.568,5.726,0.176
# "MX",  0.60636,  6.242, 0.036 -------- 0.52, 5.42, 0.15 -------- 0.63288,  6.028, 0.142, 

Fitting 10 folds for each of 9 candidates, totalling 90 fits

📊 Mejor combinación de hiperparámetros:
{'max_features': 5, 'min_samples_leaf': 1}

✅ Métricas en TRAIN:
R2: 0.93599
Pearson: 0.98
RMSE: 2.138
Bias: 0.028
Min_Pred: 3.96
Max_Pred: 69.174

✅ Métricas en TEST:
R2: 0.568
Pearson: 0.76
RMSE: 5.726
Bias: 0.176
Min_Pred: 5.047
Max_Pred: 56.009


In [10]:
# Version 03 de RF

# ============================================
# Random Forest Optimizado con RandomizedSearchCV
# ============================================

import pandas as pd
import numpy as np
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, RepeatedKFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint

# -------------------------------
# Función de evaluación
# -------------------------------
def evaluar_modelo(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_pred > 0
    y_true, y_pred = y_true[mask], y_pred[mask]
    
    r2 = r2_score(y_true, y_pred)
    pearson = np.corrcoef(y_true, y_pred)[0, 1]
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)
    
    resultados = {
        'R2': round(r2, 5),
        'Pearson': round(pearson, 3),
        'RMSE': round(rmse, 3),
        'Bias': round(bias, 3),
        'Min_Pred': round(np.min(y_pred), 3),
        'Max_Pred': round(np.max(y_pred), 3)
    }
    return resultados

# -------------------------------
# Cargar datos
# -------------------------------
estacion = "MX"
modelo = "1"
dir_base = f"D:/Josefina/Proyectos/ProyectoChile/{estacion}/modelos/ParticionDataSet/"

train_data = pd.read_csv(f"{dir_base}Modelo_{modelo}/M{modelo}_train_{estacion}.csv")
test_data = pd.read_csv(f"{dir_base}Modelo_{modelo}/M{modelo}_test_{estacion}.csv")

features = ['AOD_055','ndvi','BCSMASS_dia','DUSMASS_dia','SO4SMASS_dia',
            'v10_mean','SSSMASS_dia','blh_mean','sp_mean','SO2SMASS_dia',
            'd2m_mean','tp_mean','DEM','t2m_mean']

X_train = train_data[features]
y_train = train_data['PM25']
X_test = test_data[features]
y_test = test_data['PM25']

# -------------------------------
# Escalado opcional
# -------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------
# Definir Random Forest y parámetros aleatorios
# -------------------------------
rf = RandomForestRegressor(random_state=123, n_jobs=-1, oob_score=True)

param_dist = {
    'n_estimators': randint(500, 1001),       # entre 500 y 1000 árboles
    'max_features': randint(3, 8),            # equivalente a mtry
    'min_samples_leaf': randint(1, 6),
    'min_samples_split': randint(2, 11),
    'max_depth': [None, 20, 40],
    'bootstrap': [True, False]
}

# CV repetido 10x3
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=123)

# -------------------------------
# RandomizedSearchCV
# -------------------------------
n_iter_search = 50  # número de combinaciones aleatorias a probar

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=n_iter_search,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    verbose=2,
    random_state=123
)

# -------------------------------
# Entrenamiento
# -------------------------------
start_time = time.time()
random_search.fit(X_train_scaled, y_train)
end_time = time.time()
print(f"\n⏱️ Tiempo de entrenamiento: {end_time - start_time:.2f} segundos")

# -------------------------------
# Resultados
# -------------------------------
best_rf = random_search.best_estimator_
print("\n📊 Mejores hiperparámetros encontrados:")
print(random_search.best_params_)

# Predicciones
y_pred = best_rf.predict(X_test_scaled)
resultados_RF_cv = evaluar_modelo(y_test, y_pred)

print("\n📈 Resultados sobre el conjunto de test:")
for k, v in resultados_RF_cv.items():
    print(f"{k}: {v}")

# Parámetros finales
print("\n🌲 Parámetros del modelo final:")
print("n_estimators:", best_rf.n_estimators)
print("max_features:", best_rf.max_features)
print("min_samples_leaf:", best_rf.min_samples_leaf)
print("min_samples_split:", best_rf.min_samples_split)
print("max_depth:", best_rf.max_depth)
print("bootstrap:", best_rf.bootstrap)
print("OOB R²:", round(best_rf.oob_score_, 3))


Fitting 30 folds for each of 50 candidates, totalling 1500 fits


KeyboardInterrupt: 

In [ ]:
12:56

In [5]:
## No funciona porque hay que insalar xgboost
# # ----------------------------------------
# # XGBoost con CV=10 en Python
# # ----------------------------------------
# import pandas as pd
# import xgboost as xgb
# from sklearn.model_selection import KFold, cross_val_score
# from sklearn.metrics import r2_score, mean_squared_error
# import numpy as np
# import time

# # Función para evaluar modelo (adaptada a XGB)
# def evaluar_modelo_XGB(modelo, X_test, y_test):
#     y_pred = modelo.predict(X_test)
#     y_true, y_pred = np.array(y_test), np.array(y_pred)
#     mask = y_pred > 0
#     y_true, y_pred = y_true[mask], y_pred[mask]

#     r2 = r2_score(y_true, y_pred)
#     pearson = np.corrcoef(y_true, y_pred)[0,1]
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     bias = np.mean(y_pred - y_true)

#     resultados = {
#         'R2': round(r2,5),
#         'Pearson': round(pearson,3),
#         'RMSE': round(rmse,3),
#         'Bias': round(bias,3),
#         'Min_Pred': round(np.min(y_pred),3),
#         'Max_Pred': round(np.max(y_pred),3)
#     }
#     return resultados

# # -------------------------------
# # Cargar datos
# # -------------------------------
# estacion = "SP"
# modelo = "1"
# dir_base = f"D:/Josefina/Proyectos/ProyectoChile/{estacion}/modelos/ParticionDataSet/"

# train_data = pd.read_csv(f"{dir_base}Modelo_{modelo}/M{modelo}_train_{estacion}.csv")
# test_data = pd.read_csv(f"{dir_base}Modelo_{modelo}/M{modelo}_test_{estacion}.csv")

# features = ["AOD_055","ndvi","BCSMASS_dia","DUSMASS_dia",
#             "SO2SMASS_dia","SO4SMASS_dia","SSSMASS_dia",
#             "blh_mean","sp_mean","d2m_mean","v10_mean",
#             "u10_mean","tp_mean","DEM","dayWeek"]

# X_train = train_data[features]
# y_train = train_data['PM25']

# X_test = test_data[features]
# y_test = test_data['PM25']

# # -------------------------------
# # Convertir a DMatrix
# # -------------------------------
# dtrain = xgb.DMatrix(X_train, label=y_train)
# dtest = xgb.DMatrix(X_test, label=y_test)

# # -------------------------------
# # Parámetros del modelo
# # -------------------------------
# params = {
#     'booster': 'gbtree',
#     'objective': 'reg:squarederror',
#     'eval_metric': 'rmse',
#     'eta': 0.3,
#     'max_depth': 6,
#     'gamma': 0,
#     'subsample': 0.8,
#     'colsample_bytree': 1,
#     'min_child_weight': 1,
#     'seed': 123
# }

# # -------------------------------
# # Validación cruzada para determinar el número óptimo de rondas
# # -------------------------------
# cv_results = xgb.cv(
#     params=params,
#     dtrain=dtrain,
#     num_boost_round=2000,
#     nfold=10,
#     early_stopping_rounds=20,
#     verbose_eval=True,
#     as_pandas=True
# )

# best_nrounds = cv_results.shape[0]
# print("Número óptimo de rondas:", best_nrounds)

# # -------------------------------
# # Entrenamiento final
# # -------------------------------
# xgb_model = xgb.train(
#     params=params,
#     dtrain=dtrain,
#     num_boost_round=best_nrounds
# )

# # -------------------------------
# # Evaluación
# # -------------------------------
# resultados_XGB = evaluar_modelo_XGB(xgb_model, X_test, y_test)
# print(resultados_XGB)

# # -------------------------------
# # Hiperparámetros del modelo final
# # -------------------------------
# print("Parámetros usados:")
# print(params)
# print("Número de rondas usadas:", best_nrounds)


ModuleNotFoundError: No module named 'xgboost'

In [6]:
# Primero instalar si no lo tenés
# pip install xgboost scikit-learn pandas

import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Cargar datos
estacion = "SP"
modelo = "1"
dir_base = f"D:/Josefina/Proyectos/ProyectoChile/{estacion}/modelos/ParticionDataSet/"

train_data = pd.read_csv(f"{dir_base}Modelo_{modelo}/M{modelo}_train_{estacion}.csv")
test_data = pd.read_csv(f"{dir_base}Modelo_{modelo}/M{modelo}_test_{estacion}.csv")

X_train = train_data[["AOD_055","ndvi","BCSMASS_dia","DUSMASS_dia",
                      "SO2SMASS_dia","SO4SMASS_dia","SSSMASS_dia","blh_mean",
                      "sp_mean","d2m_mean","v10_mean","u10_mean","tp_mean","DEM","dayWeek"]]
y_train = train_data["PM25"]

X_test = test_data[["AOD_055","ndvi","BCSMASS_dia","DUSMASS_dia",
                    "SO2SMASS_dia","SO4SMASS_dia","SSSMASS_dia","blh_mean",
                    "sp_mean","d2m_mean","v10_mean","u10_mean","tp_mean","DEM","dayWeek"]]
y_test = test_data["PM25"]

# Definir modelo base
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=123)

# Opcional: hacer búsqueda de hiperparámetros reducida para no tardar tanto
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.3],
    'subsample': [0.8],
    'colsample_bytree': [1]
}

# Grid search con CV=10
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=10, scoring='r2', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# Mejor modelo
best_model = grid_search.best_estimator_
print("Mejores hiperparámetros:", grid_search.best_params_)

# Predicciones y métricas
y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
bias = np.mean(y_pred - y_test)

print(f"R2: {r2:.3f}, RMSE: {rmse:.3f}, Bias: {bias:.3f}")


ModuleNotFoundError: No module named 'xgboost'